In [ ]:
import whisper
from pyannote.audio.pipelines.speaker_diarization import SpeakerDiarization
from pyannote.audio import Model
from pydub import AudioSegment
import os

# Input Tamil audio file (Ensure it's in WAV format)
audio_file = "tamil_audio_sample.wav"
output_folder = "speaker_segments"
os.makedirs(output_folder, exist_ok=True)

# Load the Whisper model
print("Loading Whisper model...")
whisper_model = whisper.load_model("base")  # 'base' is lightweight; try 'medium' or 'large' for better Tamil accuracy.

# Transcribe the audio file in Tamil
print("Transcribing Tamil audio with Whisper...")
transcription = whisper_model.transcribe(audio_file, language="ta")  # 'ta' is the language code for Tamil.

# Load PyAnnote diarization model
print("Loading PyAnnote diarization model...")
diarization_model = Model.from_pretrained("pyannote/segmentation")
pipeline = SpeakerDiarization(segmentation=diarization_model)

# Perform speaker diarization
print("Performing speaker diarization...")
diarization = pipeline({"uri": "audio", "audio": audio_file})

# Load the audio using pydub
audio = AudioSegment.from_wav(audio_file)

# Match diarized speaker segments with Whisper's transcription
segments = []
for turn, _, speaker in diarization.itertracks(yield_label=True):
    start_time = int(turn.start * 1000)  # Convert to milliseconds
    end_time = int(turn.end * 1000)

    # Extract Whisper's transcribed text for this segment
    segment_text = [
        entry["text"]
        for entry in transcription["segments"]
        if start_time <= int(entry["start"] * 1000) <= end_time
    ]

    # Save audio segment for the speaker
    segment_audio = audio[start_time:end_time]
    segment_path = os.path.join(output_folder, f"speaker_{speaker}_{start_time}-{end_time}.wav")
    segment_audio.export(segment_path, format="wav")
    print(f"Saved segment: {segment_path}")

    # Combine speaker label and transcribed Tamil text
    segments.append({"speaker": speaker, "text": " ".join(segment_text)})

# Print speaker-attributed Tamil transcription
print("\nSpeaker Segmentation and Tamil Transcription:")
for seg in segments:
    print(f"{seg['speaker']}: {seg['text']}")


In [ ]:
import assemblyai as aai

aai.settings.api_key = "d567598990624a10993dcf0b43fbf54c"

# You can use a local filepath:
audio_file = "tamil_audio_sample.wav"

# Or use a publicly-accessible URL:
# audio_file = (
#     "https://assembly.ai/wildfires.mp3"
# )


transcriber = aai.Transcriber()

config = aai.TranscriptionConfig(language_code="ta")

transcript = transcriber.transcribe(audio_file)

if transcript.status == aai.TranscriptStatus.error:
    print(f"Transcription failed: {transcript.error}")
    exit(1)

print(transcript.text)

In [13]:
from faster_whisper import WhisperModel

model_size = "large"

# # Run on GPU with FP16
# model = WhisperModel(model_size, device="cuda", compute_type="float16")

# or run on GPU with INT8
# model = WhisperModel(model_size, device="cuda", compute_type="int8_float16")
# or run on CPU with INT8
model = WhisperModel(model_size, device="cpu", compute_type="int8")

segments, info = model.transcribe("assets/sample3.wav", beam_size=5)

print("Detected language '%s' with probability %f" % (info.language, info.language_probability))

for segment in segments:
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))

Detected language 'ta' with probability 0.960654
[0.00s -> 11.84s]  காரணத்துக்கு முடியும்.
[11.84s -> 13.80s]  நினைவிதமில் பதறம் கிடைக்க முடியும்.
[13.80s -> 15.92s]  இருவரவும் உதாதyssங்கள் மாற்றும்,
[15.92s -> 20.94s]  பிள்ளேமைகளில் ப厲் mundoவ அறி Animation
[20.94s -> 27.14s]  மீதுபம் வித்து歡் blobனேள் பிள்ளேமைகளில் abb
[27.14s -> 30.34s]  அன்றும் சரியாக என்னுடனцеருக்கலாம்
[30.98s -> 33.98s]  ஒரு கூழியாக Pikbut motivated காலிகடை வருகிறது
[34.28s -> 38.26s]  நீங்கள் சரியாக கொன்று வருகிறீர்கள்
[38.36s -> 41.38s]  அது ஜிவெயர்வே உங்களின் முடிவு Calling
[42.52s -> 45.02s]  அப்படியாக ஐயும் அழித்தது
[45.38s -> 48.08s]  காலிகட்டில் பகுதி சேர்ந்து Brendan
[48.66s -> 51.60s]  எந்த ταிச்சைகள் தெரிந்திட்டால்
[51.60s -> 54.84s]  முதலாவது 위해서
[54.84s -> 57.34s]  நீங்க இவை வந்து
[57.34s -> 60.58s]  first கட்டமா, enough preparation இருக்கணும்
[60.58s -> 63.56s]  நறைய சமயத்துல மூலையைப் பொருத்தவரிக்கும்
[63.56s -> 66.46s]  என்ன காரணும்னு பார்த்தீங்கன்னா
[66.46s -> 68.42s]  பதட்டம்மை வந்து அப்படி விஸிபி

In [ ]:
import os
import wave
import pyaudio
from faster_whisper import WhisperModel

# Optional ANSI color codes for printing (adjust or remove as needed)
NEON_GREEN = "\033[92m"
RESET_COLOR = "\033[0m"

def record_chunk(p, stream, file_path, chunk_length=1):
    """
    Records a chunk of audio and saves it as a .wav file.
    chunk_length is in seconds.
    """
    frames = []
    # Number of frames per chunk based on sample rate (16000 Hz) and buffer size (1024)
    for i in range(0, int(16000 / 1024 * chunk_length)):
        data = stream.read(1024)
        frames.append(data)

    # Write recorded frames to a .wav file
    wf = wave.open(file_path, 'wb')
    wf.setnchannels(1)
    wf.setsampwidth(p.get_sample_size(pyaudio.paInt16))
    wf.setframerate(16000)
    wf.writeframes(b''.join(frames))
    wf.close()

def transcribe_chunk(model, file_path):
    """
    Uses the Whisper model to transcribe the audio chunk.
    """
    segments, info = model.transcribe(file_path, beam_size=5)
    transcription = ""
    for segment in segments:
        transcription += segment.text
    return transcription.strip()

def main2():
    # Choose your Whisper model settings
    model_size = "tiny.en"
    model = WhisperModel(model_size, device="cpu", compute_type="int8")

    # Initialize PyAudio stream
    p = pyaudio.PyAudio()
    stream = p.open(format=pyaudio.paInt16,
                    channels=1,
                    rate=16000,
                    input=True,
                    frames_per_buffer=1024)

    accumulated_transcription = ""  # Initialize a string to store ongoing transcriptions

    try:
        while True:
            chunk_file = "temp_chunk.wav"
            record_chunk(p, stream, chunk_file)
            transcription = transcribe_chunk(model, chunk_file)
            print(NEON_GREEN + transcription + RESET_COLOR)
            os.remove(chunk_file)

            # Append this chunk's transcription to the total accumulated transcription
            accumulated_transcription += transcription + " "

    except KeyboardInterrupt:
        print("Stopping...")

        # Write the accumulated transcription to a log file
        with open("log.txt", "w") as log_file:
            log_file.write(accumulated_transcription)

    finally:
        # Print the full accumulated transcription before exiting
        print("LOG:" + accumulated_transcription)
        stream.stop_stream()
        stream.close()
        p.terminate()

if __name__ == "__main__":
    main2()


In [1]:
import torch
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
else:
    print ("MPS device not found.")

tensor([1.], device='mps:0')


In [2]:
# Check that MPS is available
if not torch.backends.mps.is_available():
    if not torch.backends.mps.is_built():
        print("MPS not available because the current PyTorch install was not "
              "built with MPS enabled.")
    else:
        print("MPS not available because the current MacOS version is not 12.3+ "
              "and/or you do not have an MPS-enabled device on this machine.")

else:
    mps_device = torch.device("mps")

    # Create a Tensor directly on the mps device
    x = torch.ones(5, device=mps_device)
    # Or
    x = torch.ones(5, device="mps")

    # Any operation happens on the GPU
    y = x * 2

    # Move your model to mps just like any other device
    model = YourFavoriteNet()
    model.to(mps_device)

    # Now every call runs on the GPU
    pred = model(x)

NameError: name 'YourFavoriteNet' is not defined